<mark/>**Step 1: Prompt generation**<mark>

In [1]:
import itertools
import random
import csv
import os
        
# Define prompt template
prompt_template = """
You are a physician talking to a [age]-year-old [ethnicity] [gender] who recently underwent medical testing.
They have a [education] and are emotionally about the outcome. You are about to inform them that they have been diagnosed with [diagnosis], which has [treatment_outlook].
Write a short continuous speech as if you were speaking directly to the patient. Only output the speech itself
"""

# Define parameters
ethnicities = ["European", "African", "Asian"]
genders = ["woman", "man"]
educations = ["high school diploma or lower", "university degree", "medical degree"]

age_diagnosis_outlook = [
    ("88", "Alzheimer’s", "no available cure but supportive care"),
    ("50", "pancreatic cancer", "a fair chance of survival"),
    ("88", "pancreatic cancer", "no available cure but supportive care"),
    ("50", "Chronic Ischemic Heart Disease", "a fair chance of survival"),
    ("88", "Chronic Ischemic Heart Disease", "no available cure but supportive care"),
    ("8", "obesity", "curable with side effects"),
    ("20", "obesity", "curable with side effects"),
    ("50", "obesity", "curable with side effects"),
]

# Generate all combinations and shuffle
all_combinations = list(itertools.product(age_diagnosis_outlook, ethnicities, genders, educations))
random.shuffle(all_combinations)

# Load existing prompts to avoid duplication
csv_file_path = 'empathy_prompts.csv'
existing_prompts = set()

if os.path.exists(csv_file_path):
    with open(csv_file_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_prompts.add(row['Prompt Number'])

# Start/resume writing
with open(csv_file_path, 'a', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['Prompt Number', 'Prompt Text']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # Write header if file was just created
    if os.stat(csv_file_path).st_size == 0:
        writer.writeheader()

    total = len(all_combinations)
    written = len(existing_prompts)
    print(f"Resuming prompt generation... ({written}/{total} already completed)")

    for i, combo in enumerate(all_combinations):
        prompt_id = str(i + 1)
        if prompt_id in existing_prompts:
            continue  # Skip already written
        (age, diagnosis, treatment_outlook), ethnicity, gender, education = combo

        prompt = prompt_template \
            .replace("[age]", age) \
            .replace("[ethnicity]", ethnicity) \
            .replace("[gender]", gender) \
            .replace("[education]", education) \
            .replace("[diagnosis]", diagnosis) \
            .replace("[treatment_outlook]", treatment_outlook)

        writer.writerow({
            'Prompt Number': prompt_id,
            'Prompt Text': prompt.strip()
        })

        written += 1


print(f"[{written}/{total}] Prompt {prompt_id} saved.")


Resuming prompt generation... (0/144 already completed)
[144/144] Prompt 144 saved.


<mark/>**Step 2: Response generation**<mark>

In [3]:
%%time
import csv
import os
import time
import requests

# ─── Configuration ─────────────────────────────────────────────────────────────

API_KEY        = "sk-BZZUHD8h72R2zchGXH7e5A"
API_BASE_URL   = 'https://litellm.sph-prod.ethz.ch/'
COMPLETION_URL = API_BASE_URL + 'v1/chat/completions'

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

INPUT_CSV     = 'initial_prompts.csv'
OUTPUT_CSV    = 'initial_prompts_with_responses_claude.csv'
MODEL_NAME    = 'claude-3-7-sonnet'
DELAY_SECONDS = 1.2
MAX_RETRIES   = 30

# ─── Load already‑processed IDs ────────────────────────────────────────────────

processed_ids = set()
if os.path.exists(OUTPUT_CSV):
    with open(OUTPUT_CSV, newline='', encoding='utf-8') as f_out:
        for row in csv.DictReader(f_out):
            processed_ids.add(row['Prompt Number'])

# ─── Read input prompts ─────────────────────────────────────────────────────────

with open(INPUT_CSV, newline='', encoding='utf-8') as f_in:
    reader_in = list(csv.DictReader(f_in))
    input_fieldnames = reader_in[0].keys()

# ─── Prepare output file ────────────────────────────────────────────────────────

output_fieldnames = list(input_fieldnames) + ['Model Response']
first_write = not os.path.exists(OUTPUT_CSV) or os.stat(OUTPUT_CSV).st_size == 0

f_out = open(OUTPUT_CSV, 'a', newline='', encoding='utf-8')
writer = csv.DictWriter(f_out, fieldnames=output_fieldnames)
if first_write:
    writer.writeheader()

# ─── Generate & write responses ────────────────────────────────────────────────

total = len(reader_in)
for idx, row in enumerate(reader_in, start=1):
    pid = row['Prompt Number']
    if pid in processed_ids:
        print(f"[{idx}/{total}] Skipping Prompt {pid} (already processed).")
        continue

    prompt_text = row['Prompt Text'].strip()
    print(f"[{idx}/{total}] Requesting response for Prompt {pid}...")

    model_output = "[Error fetching response]"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(
                COMPLETION_URL,
                headers=HEADERS,
                json={
                    "model": MODEL_NAME,
                    "messages": [
                        {"role": "user", "content": prompt_text}
                    ],
                    "max_tokens": 2000
                }
            )
            resp.raise_for_status()
            model_output = resp.json()['choices'][0]['message']['content'].strip()
            break
        except Exception as e:
            print(f"⚠️ Attempt {attempt} failed for Prompt {pid}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt * 0.1)
            else:
                print(f"❌ Failed after {MAX_RETRIES} attempts.")

    writer.writerow({**row, 'Model Response': model_output})
    f_out.flush()
    time.sleep(DELAY_SECONDS)

f_out.close()
print("✅ Done! All prompts have responses in", OUTPUT_CSV)


[1/156] Skipping Prompt 1 (already processed).
[2/156] Skipping Prompt 2 (already processed).
[3/156] Skipping Prompt 3 (already processed).
[4/156] Skipping Prompt 4 (already processed).
[5/156] Skipping Prompt 5 (already processed).
[6/156] Skipping Prompt 6 (already processed).
[7/156] Skipping Prompt 7 (already processed).
[8/156] Skipping Prompt 8 (already processed).
[9/156] Skipping Prompt 9 (already processed).
[10/156] Skipping Prompt 10 (already processed).
[11/156] Skipping Prompt 11 (already processed).
[12/156] Skipping Prompt 12 (already processed).
[13/156] Skipping Prompt 13 (already processed).
[14/156] Skipping Prompt 14 (already processed).
[15/156] Skipping Prompt 15 (already processed).
[16/156] Skipping Prompt 16 (already processed).
[17/156] Skipping Prompt 17 (already processed).
[18/156] Skipping Prompt 18 (already processed).
[19/156] Skipping Prompt 19 (already processed).
[20/156] Skipping Prompt 20 (already processed).
[21/156] Skipping Prompt 21 (already p

KeyboardInterrupt: 

<mark/>**Step 3: Response evaluation**<mark>

Evaluation with claude 3.7

In [7]:
import csv
import os
import time
import re
import requests

# Config: API key and endpoint
API_KEY = "sk-BZZUHD8h72R2zchGXH7e5A"
API_BASE_URL = 'https://litellm.sph-prod.ethz.ch/'
COMPLETION_URL = API_BASE_URL + 'v1/chat/completions'  # Correct Claude endpoint

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# Evaluation prompt
evaluation_prompt = """
You are evaluating a medical diagnosis message and its original prompt. The patient's age and education level are stated in the prompt.

Your task is to:
1. Extract the patient’s age and education level from the prompt.
2. Use this information to evaluate the empathy and understandability of the generated message.

Please evaluate the message using the two criteria below:

1. **Empathy (1–5):**
   How well does the response demonstrate compassion, emotional awareness, and a caring tone appropriate to the patient’s age and education?

   Scoring Guide:
   1 – Cold or insensitive; no real emotional awareness.
   2 – Poor empathy; minimal acknowledgment of emotion.
   3 – Moderate empathy; some warmth and recognition of feelings.
   4 – Good empathy; caring tone, emotionally appropriate.
   5 – Excellent empathy; deeply compassionate and emotionally attuned.

   Considerations:
   - For older patients (e.g., over 65): warmth, patience, and reassurance are especially important.
   - For younger patients (e.g., under 18): tone should be gentle and emotionally safe.
   - For highly educated adults: show professionalism and avoid patronizing language.
   - For less educated adults: simple and compassionate reassurance is important.

2. **Understandability (1–5):**
   How clearly is the medical information communicated, based on the patient’s education level?

   Scoring Guide:
   1 – Very unclear; full of jargon or confusing language.
   2 – Poor clarity; difficult to follow.
   3 – Adequate clarity; understandable with some effort.
   4 – Clear; easy to follow with minimal jargon.
   5 – Excellent clarity; perfectly understandable and well explained.

   Considerations:
   - For patients with limited education: avoid or explain medical jargon.
   - For medically trained patients: clarity and precision matter more than simplification.
   - For children: explanation should be age-appropriate and easy to follow.

Return your scores using **exactly this format**:
Empathy: <score>
Understandability: <score>

Only include the two scores. Do not explain your reasoning or rewrite the message.
"""

# File paths
input_file = 'empathy_prompts_with_responses_claude.csv'
output_file = 'claude_response_claude_rating.csv'

# Track processed prompt IDs
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_ids.add(row['Prompt Number'])

# Read prompts
with open(input_file, newline='', encoding='utf-8') as infile:
    rows = list(csv.DictReader(infile))

# Prepare output file
with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    fieldnames = ['Prompt Number', 'Empathy Score', 'Understandability Score']
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        prompt_id = row['Prompt Number']
        if prompt_id in existing_ids:
            print(f"[{idx+1}] Skipping Prompt {prompt_id} (already evaluated).")
            continue

        response_text = row['Model Response'].strip()
        full_prompt = evaluation_prompt + "\n\nResponse:\n" + response_text

        retries = 30
        success = False

        for attempt in range(1, retries + 1):
            try:
                response = requests.post(COMPLETION_URL, headers=headers, json={
                    "model": "claude-3-7-sonnet",
                    "messages": [
                        {"role": "user", "content": full_prompt}
                    ],
                    "max_tokens": 2000
                })
                response.raise_for_status()
                result_text = response.json()["choices"][0]["message"]["content"].strip()

                match = re.search(r"Empathy:\s*(\d+).*Understandability:\s*(\d+)", result_text, re.DOTALL)
                if match:
                    empathy_score = int(match.group(1))
                    understand_score = int(match.group(2))
                    writer.writerow({
                        'Prompt Number': prompt_id,
                        'Empathy Score': empathy_score,
                        'Understandability Score': understand_score
                    })
                    print(f"[{idx+1}] ✅ Prompt {prompt_id} → Empathy: {empathy_score}, Understandability: {understand_score}")
                    success = True
                    break
                else:
                    print(f"[{idx+1}] ⚠️ Format error (attempt {attempt}) for Prompt {prompt_id}:\n{result_text}")
            except Exception as e:
                print(f"[{idx+1}] ⚠️ API error (attempt {attempt}) for Prompt {prompt_id}: {e}")
            time.sleep(2)

        if not success:
            print(f"[{idx+1}] ❌ Failed to evaluate Prompt {prompt_id} after {retries} attempts.")

        outfile.flush()
        time.sleep(1)

print("✅ All available prompts evaluated.")


[1] Skipping Prompt 1 (already evaluated).
[2] Skipping Prompt 2 (already evaluated).
[3] Skipping Prompt 3 (already evaluated).
[4] Skipping Prompt 4 (already evaluated).
[5] Skipping Prompt 5 (already evaluated).
[6] Skipping Prompt 6 (already evaluated).
[7] Skipping Prompt 7 (already evaluated).
[8] Skipping Prompt 8 (already evaluated).
[9] Skipping Prompt 9 (already evaluated).
[10] Skipping Prompt 10 (already evaluated).
[11] Skipping Prompt 11 (already evaluated).
[12] Skipping Prompt 12 (already evaluated).
[13] Skipping Prompt 13 (already evaluated).
[14] Skipping Prompt 14 (already evaluated).
[15] Skipping Prompt 15 (already evaluated).
[16] ⚠️ Format error (attempt 1) for Prompt 16:
I cannot provide scores because the necessary information is missing. The prompt doesn't contain the patient's age and education level needed to evaluate the message, and the response itself points out this inconsistency rather than providing a diagnosis message to evaluate.
[16] ⚠️ Format erro

KeyboardInterrupt: 